In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# Animation of Light Paths in Michelson Interferometer
# simulation showing the double-pass through the Compensating Plate
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.axis('off')

#laser
ax.add_patch(plt.Rectangle((-2.8, -0.2), 0.6, 0.4, color='gray'))
ax.text(-2.5, 0.4, "Laser", ha='center', fontsize=12)

#BS
ax.plot([-0.5, 0.5], [-0.5, 0.5], color='cyan', linewidth=6, alpha=0.8)
ax.text(-0.8, -0.4, "Beam\nSplitter", ha='center', fontsize=10)

# Compensating Plate)
cp_x_offset = 1.0
ax.plot([-0.5 + cp_x_offset, 0.5 + cp_x_offset], [-0.5, 0.5], color='cyan', linewidth=6, alpha=0.4)
ax.text(cp_x_offset, 0.7, "Compensating\nPlate", ha='center', fontsize=10)

# mirror 1
ax.plot([-0.5, 0.5], [2.5, 2.5], color='silver', linewidth=8)
ax.text(0, 2.7, "Mirror 1 (Fixed)", ha='center', fontsize=12)

# mirror 2
ax.plot([2.5, 2.5], [-0.5, 0.5], color='silver', linewidth=8)
ax.text(2.6, 0.8, "Mirror 2\n(Movable)", ha='center', va='center', fontsize=12)

# پرده / سنسور (پایین)
ax.plot([-0.5, 0.5], [-2.5, -2.5], color='black', linewidth=4)
ax.text(0, -2.8, "Screen / Detector", ha='center', fontsize=12)


dot1, = ax.plot([], [], 'bo', markersize=8, label='Path 1 (Reflected)')
dot2, = ax.plot([], [], 'ro', markersize=8, label='Path 2 (Transmitted)')

ax.legend(loc='upper left', fontsize=10)

def update_paths(frame):

    total_frames = 120
    t = (frame / float(total_frames)) * 4.0  # تقسیم به ۴ فاز اصلی

    # نقاط کلیدی
    laser_pos = -2.2
    bs_pos = 0.0
    m1_pos = 2.5
    m2_pos = 2.5
    screen_pos = -2.5

    x1, y1 = laser_pos, 0
    x2, y2 = laser_pos, 0

    if t <= 1:
        dist = laser_pos + (abs(laser_pos) * t)
        x1 = x2 = dist
        y1 = y2 = 0
    elif t <= 2:
        dt = t - 1
        x1, y1 = 0, m1_pos * dt
        x2, y2 = m2_pos * dt, 0
    elif t <= 3:
        dt = t - 2
        x1, y1 = 0, m1_pos - (m1_pos * dt)
        x2, y2 = m2_pos - (m2_pos * dt), 0
    elif t <= 4:
        dt = t - 3
        x1 = x2 = 0
        y1 = y2 = screen_pos * dt

    dot1.set_data([x1], [y1])
    dot2.set_data([x2], [y2])
    return dot1, dot2

ani_setup = animation.FuncAnimation(fig, update_paths, frames=120, interval=50, blit=True)

plt.close()
HTML(ani_setup.to_jshtml())

In [ ]:
# ==========================================
# Simulation of mirror angle change dynamics
# Complete transition from circular fringes to fully linear fringes
# ==========================================

lambda_HeNe = 632.8e-9
k = 2 * np.pi / lambda_HeNe

x_screen = np.linspace(-0.01, 0.01, 500)
y_screen = np.linspace(-0.01, 0.01, 500)
X_scr, Y_scr = np.meshgrid(x_screen, y_screen)
R_scr_sq = X_scr**2 + Y_scr**2

def calculate_interference_pattern(delta_L, tilt_angle_x):
    """
    Calculate interference intensity; larger angle increase to reach fully parallel lines
    """
    f = 0.5
    phase_spherical = k * delta_L * (1 - R_scr_sq / (2 * f**2))
    phase_tilt = k * (X_scr * tilt_angle_x * 2)
    total_phase = phase_spherical + phase_tilt
    Intensity = 1 + np.cos(total_phase)
    return Intensity

fig, ax = plt.subplots(figsize=(7, 7))
ax.axis('off')

constant_delta_L = 0.001

max_tilt = 35e-5
total_frames = 150

im = ax.imshow(calculate_interference_pattern(constant_delta_L, 0),
               cmap='Reds', animated=True, extent=[-1, 1, -1, 1])

title_text = ax.set_title("Transition: Circular to Linear Fringes\nTilt Angle: 0.00e+00 rad", fontsize=14)

def update_tilt(frame_number):
    """
    In each frame, the mirror tilt angle increases until reaching the complete optical wedge state.
    """
    current_tilt = (frame_number / total_frames) * max_tilt

    im.set_array(calculate_interference_pattern(constant_delta_L, current_tilt))
    title_text.set_text(f"Transition: Circular to Linear Fringes\nMirror Tilt Angle: {current_tilt:.2e} rad")

    return [im, title_text]

animation_tilt = animation.FuncAnimation(fig, update_tilt, frames=total_frames, interval=80, blit=True)
animation_tilt.save('Transition_Circular_to_Linear_Fringes.gif', writer='pillow', fps=20)


plt.close()
HTML(animation_tilt.to_jshtml())

In [ ]:
# ==========================================
# Simulation of interference fringe dynamics (animation)
# ==========================================

fig, ax = plt.subplots(figsize=(6, 6))
ax.axis('off')

initial_path_diff = 0.001
im = ax.imshow(calculate_interference_pattern(initial_path_diff, 0),
               cmap='Reds', animated=True, extent=[-1, 1, -1, 1])
plt.title("Dynamic Fringe Shift (Moving Mirror)\n1 fringe creation = $\lambda/2$ displacement")

def update_frame(frame_number):
    """
    Update the optical path difference in each frame to simulate continuous mirror movement
    """
    current_delta_L = initial_path_diff + (frame_number * (lambda_HeNe / 10))
    im.set_array(calculate_interference_pattern(current_delta_L, 0))
    return [im]

animation_obj = animation.FuncAnimation(fig, update_frame, frames=60, interval=100, blit=True)
animation_obj.save('fringr.gif', writer='pillow', fps=20)


plt.close()
HTML(animation_obj.to_jshtml())

In [ ]:
# ==========================================
# Animation of Michelson Interferometer
# Scientific Comparison: With vs Without Compensating Plate
# Showing Glass-induced Delay and Interference Fringes
# ==========================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

def setup_axes(ax, title, show_plate=True):
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3.5, 3)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

    # Laser
    ax.add_patch(plt.Rectangle((-2.8, -0.2), 0.6, 0.4, color='gray', zorder=2))
    ax.text(-2.5, 0.4, "Laser", ha='center', fontsize=10)

    # Beam splitter
    ax.add_patch(plt.Rectangle((0.212, -0.354), 0.2, 0.8, angle=45, color='lightblue', alpha=0.6, label='Glass'))
    ax.plot([-0.35, 0.35], [-0.35, 0.35], color='silver', linewidth=2)
    ax.text(-0.8, -0.6, "Beam\nSplitter", ha='center', fontsize=9)

    # Compensating plate
    if show_plate:
        # Position in the movable beam path
        ax.add_patch(plt.Rectangle((0.8, -0.4), 0.2, 0.8, angle=45, color='lightblue', alpha=0.6))
        ax.text(1.2, 0.7, "Compensating\nPlate", ha='center', fontsize=9, color='blue')

    # Mirror 1 (fixed - top)
    ax.plot([-0.5, 0.5], [2.5, 2.5], color='black', linewidth=6)
    ax.text(0, 2.8, "Mirror 1", ha='center', fontsize=10)

    # Mirror 2 (movable - right)
    ax.plot([2.5, 2.5], [-0.5, 0.5], color='black', linewidth=6)
    ax.text(2.8, 0.8, "Mirror 2", ha='center', va='center', fontsize=10)

    # Screen / sensor
    ax.plot([-0.7, 0.7], [-2.8, -2.8], color='darkred', linewidth=4)
    ax.text(0, -3.1, "Screen", ha='center', fontsize=11, fontweight='bold')

setup_axes(ax1, "With Compensating Plate (Symmetric)", show_plate=True)
setup_axes(ax2, "Without Compensating Plate (Asymmetric)", show_plate=False)

# Define beams
dot1_a, = ax1.plot([], [], 'bo', markersize=9, label='Path 1 (Vertical)', zorder=5)
dot2_a, = ax1.plot([], [], 'ro', markersize=9, label='Path 2 (Horizontal)', zorder=5)
dot1_b, = ax2.plot([], [], 'bo', markersize=9)
dot2_b, = ax2.plot([], [], 'ro', markersize=9)

def calculate_pos(time, is_path2, has_cp):
    v_air = 2.5
    n_glass = 1.6
    v_glass = v_air / n_glass
    d_glass = 0.4

    # Path from laser to center
    t_to_bs = 2.2 / v_air
    if time <= t_to_bs:
        return -2.2 + v_air * time, 0

    # Remaining time after reaching the center
    t_rem = time - t_to_bs

    # Calculate the time needed to traverse the arms (round trip = 5 units)
    # In the horizontal path (Path 2) we always pass through glass twice
    # In the vertical path (Path 1) we pass through glass twice only if the plate is present
    if is_path2:
        glass_passes = 2
    else:
        glass_passes = 2 if has_cp else 0

    t_glass = (glass_passes * d_glass) / v_glass
    t_air_in_arm = (5.0 - (glass_passes * d_glass)) / v_air
    t_total_arm = t_glass + t_air_in_arm

    # If still in the arms
    if t_rem <= t_total_arm:
        # Average speed in the arm for animation simplicity
        v_eff = 5.0 / t_total_arm
        dist = v_eff * t_rem
        if dist <= 2.5: # going toward the mirror
            return (dist, 0) if is_path2 else (0, dist)
        else: # returning from the mirror
            return (2.5 - (dist - 2.5), 0) if is_path2 else (0, 2.5 - (dist - 2.5))

    # Path from center to screen
    t_rem_final = t_rem - t_total_arm
    dist_screen = v_air * t_rem_final
    if dist_screen > 2.8: dist_screen = 2.8
    return 0, -dist_screen

def update(frame):
    t = frame / 40.0

    p1a = calculate_pos(t, False, True)
    p2a = calculate_pos(t, True, True)
    p1b = calculate_pos(t, False, False)
    p2b = calculate_pos(t, True, False)

    dot1_a.set_data([p1a[0]], [p1a[1]])
    dot2_a.set_data([p2a[0]], [p2a[1]])
    dot1_b.set_data([p1b[0]], [p1b[1]])
    dot2_b.set_data([p2b[0]], [p2b[1]])

    # Display fringes at the end of the path
    # In the symmetric case (left side) both arrive at the same time
    if p1a[1] <= -2.8 and p2a[1] <= -2.8:
        if len(ax1.images) == 0:
            x = np.linspace(-0.6, 0.6, 100)
            Z = np.sin(20 * x)**2
            ax1.imshow(Z.reshape(1, -1), extent=[-0.6, 0.6, -3.0, -2.8], cmap='Reds', aspect='auto', zorder=1)

    # In the asymmetric case (right side) the beams arrive with a time difference
    if p1b[1] <= -2.8 and p2b[1] <= -2.8:
        if len(ax2.images) == 0:
            # Faded interference pattern due to the absence of the compensating plate
            Z = np.ones((1, 100)) * 0.3
            ax2.imshow(Z, extent=[-0.6, 0.6, -3.0, -2.8], cmap='Reds', aspect='auto', zorder=1, vmin=0, vmax=1)

    return dot1_a, dot2_a, dot1_b, dot2_b

ani = animation.FuncAnimation(fig, update, frames=300, interval=30, blit=True)
ani.save('michelson_fixed.gif', writer='pillow', fps=30)

plt.tight_layout()
plt.show()
HTML(ani.to_jshtml())